In [1]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april import Evaluator
from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

In [2]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

2025-03-20 15:18:54.849424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742480334.862368   37481 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742480334.866658   37481 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742480334.877907   37481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742480334.877932   37481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742480334.877934   37481 computation_placer.cc:177] computation placer alr

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


In [3]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()

In [11]:
datasets = sorted([e.name for e in get_event_log_files() if e.p == 0.3])
# select_datasets = ["paper", "p2p", "small", "medium"]
select_datasets = ["p2p"]
select_attributes = ["-1"]
datasets = [d for d in datasets if any([s in d for s in select_datasets])]
datasets = [d for d in datasets if any([s in d for s in select_attributes])]
dataset_name = datasets[0]
print(datasets)


['p2p-0.3-1']


In [7]:
from april.anomalydetection.ltnencoder import LTNDAEP2P

In [8]:

ldp =  LTNDAEP2P()

In [9]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april import Evaluator
from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

In [12]:

dataset = Dataset(dataset_name)


In [22]:
flat_onehot_features_2d = dataset.flat_onehot_features_2d
flat_features = dataset.flat_features
features  = dataset.features
print(f"features.shape: {len(features)}")
print(f"flat_features.shape: {flat_features.shape}")
print(f"flat_onehot_features_2d.shape: {flat_onehot_features_2d.shape}")


features.shape: 2
flat_features.shape: (5000, 16, 2)
flat_onehot_features_2d.shape: (5000, 2688)


In [ ]:
print(features[0][0])
print(features[1][0])


[27.  7. 11.  4.  5. 24.  9. 10.  8. 26.  0.  0.  0.  0.  0.  0.]
[141.  32.  51.  27. 121.  13.  49. 123.  40. 140.   0.   0.   0.   0.
   0.   0.]


In [47]:
print(flat_features[0])
print(type(flat_features))

[[ 27. 141.]
 [  7.  32.]
 [ 11.  51.]
 [  4.  27.]
 [  5. 121.]
 [ 24.  13.]
 [  9.  49.]
 [ 10. 123.]
 [  8.  40.]
 [ 26. 140.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]]
<class 'numpy.ndarray'>


In [ ]:
print(flat_onehot_features_2d[0])

In [ ]:
# ads = [
#     dict(ad=DAE, fit_kwargs=dict(epochs=30, batch_size=500))
# ]
# for ad in ads:
#     [fit_and_save(d, **ad) for d in tqdm(datasets, leave=True, position=1)]

DAE:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - loss: 0.2488 - val_loss: 0.2448
Epoch 2/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.2415 - val_loss: 0.2351
Epoch 3/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.2287 - val_loss: 0.2158
Epoch 4/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.2052 - val_loss: 0.1804
Epoch 5/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1659 - val_loss: 0.1261
Epoch 6/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1109 - val_loss: 0.0655
Epoch 7/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0564 - val_loss: 0.0255
Epoch 8/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0230 - val_loss: 0.0105
Epoch 9/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0105 - val_loss: 0.0063
Epoch 10/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0066 - val_loss: 0.0051
Epoch 11/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0055 - val_loss: 0.0047
Epoch 12/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0050 - val_loss: 0.0045


In [7]:
dataset = Dataset(datasets[0])

In [8]:
print(dataset)
print(vars(dataset))

{'dataset_name': 'small-0.3-1', 'go_backwards': False, 'pad_mode': 'post', 'attribute_types': [<AttributeType.CATEGORICAL: 0>, <AttributeType.CATEGORICAL: 0>], 'attribute_keys': ['name', 'user'], 'classes': array([[[0, 0],
        [0, 0],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [2, 9],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]],

       ...,

       [[0, 0],
        [0, 0],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]],

       [[0, 0],
        [0, 0],
        [0, 0],
        ...,
        [0, 0],
        [0, 0],
        [0, 0]]]), 'labels': array(['normal', {'anomaly': 'Insert', 'attr': {'indices': [0, 5]}},
       'normal', ...,
       {'ano